# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [3]:
#your code here
spaceship.shape

(8693, 14)

**Check for data types**

In [4]:
#your code here
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [5]:
#your code here
spaceship.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [6]:
#your code here
# Drop rows with any missing values
spaceship_clean = spaceship.dropna().reset_index(drop=True)

# Quick check
spaceship_clean.shape, spaceship_clean.isna().sum().sum()

((6606, 14), 0)

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [7]:
#your code here
# Extract deck letter from "Cabin" (e.g., "B/123/S" -> "B")
spaceship_clean["Deck"] = spaceship_clean["Cabin"].str.split("/").str[0]

# (optional) ensure only expected values remain
allowed = set(list("ABCDEFGT"))
spaceship_clean["Deck"] = spaceship_clean["Deck"].where(spaceship_clean["Deck"].isin(allowed))

# quick check
spaceship_clean["Deck"].unique()

array(['B', 'F', 'A', 'G', 'E', 'C', 'D', 'T'], dtype=object)

- Drop PassengerId and Name

In [8]:
#your code here
# Drop PassengerId and Name (robust to casing/typos)
to_drop = [c for c in ["PassengerId", "Name", "passengerld", "name"] if c in spaceship_clean.columns]
spaceship_clean = spaceship_clean.drop(columns=to_drop)

spaceship_clean.columns.tolist()  # quick check

['HomePlanet',
 'CryoSleep',
 'Cabin',
 'Destination',
 'Age',
 'VIP',
 'RoomService',
 'FoodCourt',
 'ShoppingMall',
 'Spa',
 'VRDeck',
 'Transported',
 'Deck']

- For non-numerical columns, do dummies.

In [9]:
#your code here
# One-hot encode non-numeric columns (keep target 'Transported' separate/int)
df = spaceship_clean.copy()

# (optional) drop original Cabin now that Deck is created
if "Cabin" in df.columns:
    df = df.drop(columns=["Cabin"])

# Convert boolean cols to 0/1 (KNN needs numeric)
bool_cols = df.select_dtypes(include="bool").columns.drop("Transported", errors="ignore")
df[bool_cols] = df[bool_cols].astype(int)

# One-hot encode object/category columns (exclude target if present)
cat_cols = df.select_dtypes(include=["object", "category"]).columns.drop("Transported", errors="ignore")
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)  # drop_first optional

# Keep target as numeric (0/1)
if "Transported" in df.columns:
    df["Transported"] = df["Transported"].astype(int)

# Quick checks
print("Shape after dummies:", df.shape)
print("Non-numeric columns remaining:", df.select_dtypes(exclude=[np.number]).columns.tolist())

Shape after dummies: (6606, 20)
Non-numeric columns remaining: ['HomePlanet_Europa', 'HomePlanet_Mars', 'CryoSleep_True', 'Destination_PSO J318.5-22', 'Destination_TRAPPIST-1e', 'VIP_True', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T']


**Perform Train Test Split**

In [10]:
#your code here
from sklearn.model_selection import train_test_split

X = df.drop(columns="Transported")
y = df["Transported"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((5284, 19), (1322, 19), (5284,), (1322,))

**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [11]:
#your code here
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

# KNN needs scaling — build a pipeline
pipe = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier()
)

# Hyperparameter search space
param_grid = {
    "kneighborsclassifier__n_neighbors": list(range(3, 31, 2)),  # odd ks
    "kneighborsclassifier__weights": ["uniform", "distance"],
    "kneighborsclassifier__p": [1, 2],  # Manhattan vs Euclidean
}

# Grid search with 5-fold CV (optimize accuracy; change to 'f1' if desired)
gs = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gs.fit(X_train, y_train)

best_knn = gs.best_estimator_
print("Best params:", gs.best_params_)
print("CV best accuracy:", gs.best_score_)
print("Test accuracy:", best_knn.score(X_test, y_test))

Best params: {'kneighborsclassifier__n_neighbors': 23, 'kneighborsclassifier__p': 1, 'kneighborsclassifier__weights': 'uniform'}
CV best accuracy: 0.7833116524755599
Test accuracy: 0.7904689863842662


c:\Users\ruchi\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\ruchi\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\ruchi\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ruchi\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\ruchi\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

- Evaluate your model's performance. Comment it

In [12]:
#your code here
thr = 0.4  # example
y_pred_thr = (y_proba >= thr).astype(int)

NameError: name 'y_proba' is not defined